### Transform Races Data
1. Read bronze races table
2. Keep only the columns required for analytics (Drop URL column)
3. Standarize columns name using snake_case (raceName -> race_name,circuitId -> circuit_id)
4. Rename columns to make them more meaningful (date -> race_date)
5. Remove duplicate records
6. Transform value of column race_name to Title Case
7. Write the transformed data to silver races table

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"

##### 1. Read bronze races table

In [0]:
races_df = spark.read.table(bronze_table)

In [0]:
display(races_df)

##### 2. Keep only the columns required for analytics (Drop URL column)


In [0]:
races_selected_columns = races_df.select(
    "season",
    "round",
    "raceName",
    "date",
    "circuitId",
    "ingestion_timestamp",
    "source_file"
)

In [0]:
from pyspark.sql import functions as F

In [0]:
races_column_renamed_df = (
    races_selected_columns
    .withColumnRenamed("raceName","race_name")
    .withColumnRenamed("circuitId","circuit_id")
    .withColumnRenamed("date","race_date")
)

In [0]:
races_distinct_df = races_column_renamed_df.dropDuplicates(['season','round'])

In [0]:
display(races_distinct_df)

In [0]:
races_final_df = (
    races_distinct_df.withColumn("race_name",F.initcap(F.col("race_name")))
)

In [0]:
display(races_final_df)

In [0]:
(
    races_final_df.write.mode("overwrite").saveAsTable(silver_table)
)